# Analysis A — Postprocessing notebook
Short notebook to load outputs, produce summary figures, and run statistical tests.

In [ ]:
# Section 1: Importar librerías
import os
import sys
from pathlib import Path
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from itertools import combinations
# Optional imports with graceful fallback
try:
    import scipy.stats as stats
except Exception as e:
    stats = None
    logging.warning('scipy not available: %s', e)
# Configure seaborn
sns.set(style='whitegrid', rc={'figure.dpi': 120})

In [ ]:
# Section 2: Configurar entorno y semilla
BASE_DIR = Path(__file__).parent
OUT_DIR = BASE_DIR / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
print('Paths: ', OUT_DIR, FIG_DIR)

In [ ]:
# Section 3: Cargar datos
hv_csv = OUT_DIR / 'hv_series.csv'
conv_csv = OUT_DIR / 'convergence_times.csv'
metrics_csv = OUT_DIR / 'metrics_summary.csv'
print('Looking for:', hv_csv, conv_csv, metrics_csv)
df_hv = pd.read_csv(hv_csv) if hv_csv.exists() else pd.DataFrame()
df_conv = pd.read_csv(conv_csv) if conv_csv.exists() else pd.DataFrame()
df_metrics = pd.read_csv(metrics_csv) if metrics_csv.exists() else pd.DataFrame()
print('Loaded: hv rows', len(df_hv), 'conv rows', len(df_conv), 'metrics rows', len(df_metrics))
display_vars = {'hv': df_hv.head(3), 'conv': df_conv.head(3), 'metrics': df_metrics.head(3)}
for k,v in display_vars.items():
    print('\n==',k,'==')
    display(v)

In [ ]:
# Section 4: Preprocesamiento y limpieza
def clean_df_hv(df):
    df2 = df.copy()
    # ensure types
    df2['eval'] = pd.to_numeric(df2['eval'], errors='coerce').astype('Int64')
    df2['hv'] = pd.to_numeric(df2['hv'], errors='coerce')
    df2 = df2.dropna(subset=['eval','hv'])
    return df2

def clean_df_conv(df):
    df2 = df.copy()
    df2['convergence_eval'] = pd.to_numeric(df2['convergence_eval'], errors='coerce').astype('Int64')
    df2 = df2.dropna(subset=['convergence_eval'])
    return df2

df_hv = clean_df_hv(df_hv)
df_conv = clean_df_conv(df_conv)
print('After cleaning: hv rows', len(df_hv), 'conv rows', len(df_conv))

In [ ]:
# Section 5: Funciones y utilidades reutilizables
def save_fig(fig, name):
    p = FIG_DIR / name
    fig.savefig(p, bbox_inches='tight', dpi=200)
    print('Saved', p)
    return p

def plot_convergence_boxplots(df_conv):
    problems = df_conv['problem'].unique()
    figs = {}
    for prob in problems:
        sub = df_conv[df_conv['problem']==prob]
        plt.figure(figsize=(6,4))
        sns.boxplot(data=sub, x='budget', y='convergence_eval')
        plt.title(f'Convergence times — {prob}')
        plt.ylabel('Evaluations to 95%')
        p = FIG_DIR / f'convergence_box_{prob}.png'
        plt.savefig(p, bbox_inches='tight', dpi=200)
        plt.close()
        figs[prob] = p
    return figs

def plot_auc_bars(df_metrics):
    figs = {}
    for prob in df_metrics['problem'].unique():
        sub = df_metrics[df_metrics['problem']==prob]
        plt.figure(figsize=(6,4))
        sns.barplot(data=sub, x='budget', y='AUC_median')
        plt.title(f'AUC median — {prob}')
        p = FIG_DIR / f'auc_bar_{prob}.png'
        plt.savefig(p, bbox_inches='tight', dpi=200)
        plt.close()
        figs[prob] = p
    return figs

def pairwise_wilcoxon(df_conv):
    # pairwise Wilcoxon between budgets for each problem on convergence_eval
    rows = []
    for prob in df_conv['problem'].unique():
        sub = df_conv[df_conv['problem']==prob]
        budgets = sorted(sub['budget'].unique())
        for a,b in combinations(budgets,2):
            sa = sub[sub['budget']==a]['convergence_eval'].dropna()
            sb = sub[sub['budget']==b]['convergence_eval'].dropna()
            if len(sa)>=5 and len(sb)>=5 and stats is not None:
                try:
                    stat, p = stats.wilcoxon(sa.values, sb.values)
                except Exception:
                    stat, p = float('nan'), float('nan')
            else:
                stat, p = float('nan'), float('nan')
            rows.append({'problem':prob,'budget_a':a,'budget_b':b,'stat':stat,'p_value':p,'n_a':len(sa),'n_b':len(sb)})
    return pd.DataFrame(rows)

In [ ]:
# Section 6: Exploración y visualización (rápida)
print('Convergence summary:')
display(df_conv.groupby(['problem','budget'])['convergence_eval'].describe())
print('\nMetrics summary:')
display(df_metrics)

In [ ]:
# Section 7: Implementación del flujo principal — generar figuras
figs1 = plot_convergence_boxplots(df_conv)
figs2 = plot_auc_bars(df_metrics)
print('Generated figures:')
for k,v in {**figs1, **figs2}.items():
    print(k, v)

In [ ]:
# Section 8: Pruebas unitarias integradas (simples)
def _test_plot_funcs():
    assert len(df_conv)>0 or len(df_metrics)>0, 'No data available for tests'
    # call plotting functions in-memory (do not overwrite)
    _ = plot_convergence_boxplots(df_conv) if len(df_conv)>0 else {}
    _ = plot_auc_bars(df_metrics) if len(df_metrics)>0 else {}
    print('Sanity tests passed')

_test_plot_funcs()

In [ ]:
# Section 9: Ejecución, registros y salida (pruebas estadísticas)
if stats is None:
    print('scipy not available — skipping statistical tests. If you want them, install scipy.')
else:
    print('Running pairwise Wilcoxon tests...')
    df_stats = pairwise_wilcoxon(df_conv)
    out_stats = OUT_DIR / 'stat_tests.csv'
    df_stats.to_csv(out_stats, index=False)
    print('Saved stat tests to', out_stats)
    display(df_stats.head())

# Section 10: Guardar artefactos y nota final
print('Figures folder:', FIG_DIR)
print('Outputs folder:', OUT_DIR)
print('Notebook complete. Run cells sequentially from top to bottom in VS Code.')